# Crypto OHLCV Downloader - 1h
Downloads **1h** klines from `data.binance.vision` (USDT-M perpetuals: monthly archives + daily archives for the ongoing month), tops up the latest closed bars via Binance REST / `ccxt`, and saves one Parquet file per symbol in Google Drive.

Run the cells top to bottom. Re-running is incremental (it only re-pulls the last month and anything newer).

## 1. Install & mount Drive

In [1]:
!pip install -q ccxt pyarrow python-dateutil requests pandas



[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: C:\Users\Kaizen\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Running in local environment (Google Colab drive mount skipped).")

Running in local environment (Google Colab drive mount skipped).


## 2. Config

In [3]:
import os
# ============ CONFIG (edit here) ============
TF          = "1h"                 # timeframe for this notebook
SYMBOLS     = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT"]
START_DATE  = "2020-01-01"             # earliest date to download (YYYY-MM-DD)
END_DATE    = None                     # latest date to download (None = today e.g. 2026-10-25)
MARKET      = "futures/um"             # USDT-M perpetual (per spec); use "spot" for spot
SAVE_CSV    = False                    # parquet is always saved; set True to also save CSV
DEFAULT_DIR = "/content/drive/MyDrive/crypto_data" if ("IN_COLAB" in globals() and IN_COLAB) else r"D:\Projects_Main\Trade-Pro\Trader-Pro\notebooks\data"
DRIVE_DIR   = os.environ.get("CRYPTO_DATA_DIR", DEFAULT_DIR)   # data folder: D:\Projects_Main\Trade-Pro\Trader-Pro\notebooks\data
USE_TOPUP   = True                     # fetch today's closed bars via Binance REST API / CCXT
MAX_WORKERS = 8
# ============================================

## 3. Helpers

In [4]:
import os, io, zipfile, time
import requests
import pandas as pd
from datetime import datetime, timezone, timedelta
from dateutil.relativedelta import relativedelta
from concurrent.futures import ThreadPoolExecutor

OUT_DIR = os.path.join(DRIVE_DIR, TF)
os.makedirs(OUT_DIR, exist_ok=True)

COLS = ["open_time","open","high","low","close","volume","close_time",
        "quote_volume","trades","taker_buy_base","taker_buy_quote","ignore"]
KEEP = ["open_time","open","high","low","close","volume","close_time",
        "quote_volume","trades","taker_buy_base","taker_buy_quote"]
BASE_M = f"https://data.binance.vision/data/{MARKET}/monthly/klines"
BASE_D = f"https://data.binance.vision/data/{MARKET}/daily/klines"
TF_DELTA = {"15m": pd.Timedelta(minutes=15), "1h": pd.Timedelta(hours=1),
            "4h": pd.Timedelta(hours=4), "1d": pd.Timedelta(days=1),
            "1w": pd.Timedelta(days=7)}[TF]
print("Output folder:", OUT_DIR)

Output folder: D:\Projects_Main\Trade-Pro\Trader-Pro\notebooks\data\1h


In [5]:
def normalize(df):
    """Clean a raw Binance kline dataframe -> typed, UTC-indexed-ready frame."""
    df = df.copy()
    df = df[pd.to_numeric(df["open_time"], errors="coerce").notna()]   # drop header rows
    for c in COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    # newer files can be in microseconds; convert to milliseconds
    for c in ["open_time", "close_time"]:
        if c in df.columns:
            df.loc[df[c] > 1e14, c] = df.loc[df[c] > 1e14, c] // 1000
    df["open_time"]  = pd.to_datetime(df["open_time"].astype("int64"),  unit="ms", utc=True)
    df["close_time"] = pd.to_datetime(df["close_time"].astype("int64"), unit="ms", utc=True)
    if "trades" in df.columns:
        df["trades"] = df["trades"].fillna(0).astype("int64")
    for c in KEEP:
        if c not in df.columns:
            df[c] = float("nan")
    return df[KEEP]


def fetch_zip(url):
    """Download and unzip one kline zip file from Binance Vision. Returns df or None."""
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 404:
                return None
            r.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                with z.open(z.namelist()[0]) as f:
                    raw = pd.read_csv(f, header=None, names=COLS)
            return normalize(raw)
        except Exception as e:
            if attempt == 2:
                return None
            time.sleep(1.5 * (attempt + 1))
    return None


def fetch_month(symbol, tf, year, month):
    """Download one monthly zip from data.binance.vision."""
    ym = f"{year}-{month:02d}"
    url = f"{BASE_M}/{symbol}/{tf}/{symbol}-{tf}-{ym}.zip"
    return fetch_zip(url)


def fetch_day(symbol, tf, date_str):
    """Download one daily zip from data.binance.vision."""
    url = f"{BASE_D}/{symbol}/{tf}/{symbol}-{tf}-{date_str}.zip"
    return fetch_zip(url)


def rest_topup(symbol, last_open, tf=TF, market=MARKET):
    """Fetch recent/today's closed candles via Binance public REST API. Falls back to CCXT if needed."""
    endpoint = "https://fapi.binance.com/fapi/v1/klines" if market == "futures/um" else "https://api.binance.com/api/v3/klines"
    dur_map = {"15m": 15*60*1000, "1h": 60*60*1000, "4h": 4*60*60*1000, "1d": 24*60*60*1000, "1w": 7*24*60*60*1000}
    dur = dur_map.get(tf, 15*60*1000)
    since = int(last_open.timestamp() * 1000) + 1
    rows = []
    while True:
        params = {"symbol": symbol, "interval": tf, "startTime": since, "limit": 1000}
        try:
            r = requests.get(endpoint, params=params, timeout=15)
            if r.status_code != 200:
                break
            batch = r.json()
            if not batch or not isinstance(batch, list):
                break
            rows += batch
            since = batch[-1][0] + 1
            if len(batch) < 2:
                break
        except Exception:
            break

    # Fallback to ccxt if direct REST fails / geo-blocked
    if not rows:
        try:
            import ccxt
            ex = ccxt.binanceusdm({"enableRateLimit": True}) if market == "futures/um" else ccxt.binance({"enableRateLimit": True})
            sym = symbol.replace("USDT", "/USDT:USDT") if market == "futures/um" else symbol.replace("USDT", "/USDT")
            since_ccxt = int(last_open.timestamp() * 1000) + 1
            while True:
                batch = ex.fetch_ohlcv(sym, tf, since=since_ccxt, limit=1000)
                if not batch:
                    break
                for b in batch:
                    ct = b[0] + dur - 1
                    qv = b[5] * (b[1] + b[2] + b[3] + b[4]) / 4
                    rows.append([b[0], b[1], b[2], b[3], b[4], b[5], ct, qv, 0, float("nan"), float("nan"), 0])
                since_ccxt = batch[-1][0] + 1
                if len(batch) < 2:
                    break
        except Exception:
            pass

    now_ms = int(time.time() * 1000)
    rows = [r for r in rows if r[0] + dur <= now_ms]       # closed candles only
    if not rows:
        return None
    raw = pd.DataFrame(rows, columns=COLS[:len(rows[0])])
    return normalize(raw)

## 4. Download & save

In [6]:
def process_symbol(symbol):
    path = os.path.join(OUT_DIR, f"{symbol}_{TF}.parquet")
    existing = pd.read_parquet(path) if os.path.exists(path) else None

    # Calculate date bounds
    start_dt = datetime.fromisoformat(START_DATE).replace(tzinfo=timezone.utc)
    if existing is not None and len(existing):
        start_dt = existing["open_time"].iloc[-1].to_pydatetime() - relativedelta(months=1)
        start_dt = max(start_dt, datetime.fromisoformat(START_DATE).replace(tzinfo=timezone.utc))

    if END_DATE is not None:
        end_dt = datetime.fromisoformat(END_DATE).replace(tzinfo=timezone.utc)
    else:
        end_dt = datetime.now(timezone.utc)

    # 1. Monthly files for all completed full months
    last_full_month_dt = (end_dt.replace(day=1) - relativedelta(days=1))
    cur = start_dt.replace(day=1)
    months = []
    while cur <= last_full_month_dt:
        months.append((cur.year, cur.month))
        cur += relativedelta(months=1)

    print(f"[{symbol}] downloading {len(months)} monthly files ({start_dt.strftime('%Y-%m')} -> {last_full_month_dt.strftime('%Y-%m')}) ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        m_parts = list(ex.map(lambda ym: fetch_month(symbol, TF, *ym), months))
    parts = [p for p in m_parts if p is not None and len(p)]

    # 2. Daily files for the current/incomplete month up to yesterday
    day_cur = max(start_dt.date(), end_dt.replace(day=1).date())
    yesterday_utc = (datetime.now(timezone.utc) - timedelta(days=1)).date()
    if END_DATE is not None:
        day_end = min(end_dt.date(), yesterday_utc)
    else:
        day_end = yesterday_utc

    days = []
    while day_cur <= day_end:
        days.append(day_cur.strftime("%Y-%m-%d"))
        day_cur += timedelta(days=1)

    if days:
        print(f"[{symbol}] downloading {len(days)} daily files ({days[0]} -> {days[-1]}) ...")
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            d_parts = list(ex.map(lambda d: fetch_day(symbol, TF, d), days))
        parts.extend([p for p in d_parts if p is not None and len(p)])

    if existing is not None:
        parts.insert(0, existing)

    if not parts:
        print(f"[{symbol}] no data found (symbol may not exist for this market/date).")
        return None

    df = pd.concat(parts, ignore_index=True)
    df = df.drop_duplicates("open_time").sort_values("open_time").reset_index(drop=True)

    # 3. Top-up today's closed bars via REST API / CCXT if requested
    if USE_TOPUP and (END_DATE is None or end_dt.date() >= datetime.now(timezone.utc).date()):
        try:
            extra = rest_topup(symbol, df["open_time"].iloc[-1], TF, MARKET)
            if extra is not None and len(extra):
                df = pd.concat([df, extra]).drop_duplicates("open_time").sort_values("open_time").reset_index(drop=True)
                print(f"[{symbol}] top-up added {len(extra)} bars up to {df['open_time'].iloc[-1]}")
        except Exception as e:
            print(f"[{symbol}] top-up skipped ({type(e).__name__}: {str(e)[:120]})")

    # Filter to END_DATE if explicitly specified
    if END_DATE is not None:
        cutoff = pd.to_datetime(END_DATE, utc=True) + pd.Timedelta(days=1)
        df = df[df["open_time"] < cutoff].reset_index(drop=True)

    df.to_parquet(path, index=False)
    if SAVE_CSV:
        df.to_csv(path.replace(".parquet", ".csv"), index=False)
    print(f"[{symbol}] saved {len(df):,} bars ({df['open_time'].iloc[0]} -> {df['open_time'].iloc[-1]})")
    return df


results = {}
for s in SYMBOLS:
    results[s] = process_symbol(s)
print("\nDone.")

[BTCUSDT] downloading 1 monthly files (2026-09 -> 2026-09) ...
[BTCUSDT] downloading 26 daily files (2026-10-01 -> 2026-10-26) ...
[BTCUSDT] top-up added 28 bars up to 2026-10-27 03:00:00+00:00
[BTCUSDT] saved 59,788 bars (2020-01-01 00:00:00+00:00 -> 2026-10-27 03:00:00+00:00)
[ETHUSDT] downloading 1 monthly files (2026-09 -> 2026-09) ...
[ETHUSDT] downloading 26 daily files (2026-10-01 -> 2026-10-26) ...
[ETHUSDT] top-up added 28 bars up to 2026-10-27 03:00:00+00:00
[ETHUSDT] saved 59,788 bars (2020-01-01 00:00:00+00:00 -> 2026-10-27 03:00:00+00:00)
[SOLUSDT] downloading 1 monthly files (2026-09 -> 2026-09) ...
[SOLUSDT] downloading 26 daily files (2026-10-01 -> 2026-10-26) ...
[SOLUSDT] top-up added 28 bars up to 2026-10-27 03:00:00+00:00
[SOLUSDT] saved 53,493 bars (2020-09-14 07:00:00+00:00 -> 2026-10-27 03:00:00+00:00)
[BNBUSDT] downloading 1 monthly files (2026-09 -> 2026-09) ...
[BNBUSDT] downloading 26 daily files (2026-10-01 -> 2026-10-26) ...
[BNBUSDT] top-up added 28 bars u

## 5. Quality check

In [7]:
# ---- Data quality report ----
rows = []
for s, df in results.items():
    if df is None or df.empty:
        rows.append([s, 0, None, None, None, None]); continue
    diffs = df["open_time"].diff().dropna()
    n_gaps = int((diffs != TF_DELTA).sum())
    max_gap = diffs.max() if len(diffs) else None
    rows.append([s, len(df), df["open_time"].iloc[0], df["open_time"].iloc[-1], n_gaps, max_gap])
report = pd.DataFrame(rows, columns=["symbol","bars","first_bar","last_bar","irregular_steps","largest_gap"])
display(report)

# peek at one file
first = next((d for d in results.values() if d is not None), None)
if first is not None:
    display(first.tail())

,symbol,bars,first_bar,last_bar,irregular_steps,largest_gap
0,BTCUSDT,59788,2020-01-01 00:00:00+00:00,2026-10-27 03:00:00+00:00,0,0 days 01:00:00
1,ETHUSDT,59788,2020-01-01 00:00:00+00:00,2026-10-27 03:00:00+00:00,0,0 days 01:00:00
2,SOLUSDT,53493,2020-09-14 07:00:00+00:00,2026-10-27 03:00:00+00:00,2,3 days 01:00:00
3,BNBUSDT,58820,2020-02-10 08:00:00+00:00,2026-10-27 03:00:00+00:00,0,0 days 01:00:00
4,XRPUSDT,59540,2020-01-06 08:00:00+00:00,2026-10-27 03:00:00+00:00,2,3 days 01:00:00


,open_time,open,high,low,close,volume,close_time,quote_volume,trades,taker_buy_base,taker_buy_quote
59783,2026-10-26 23:00:00+00:00,79977.9,80156.0,79684.8,80115.9,3144.470,2026-10-26 23:59:59.999000+00:00,2.512146e+08,81022,1435.324,1.147017e+08
59784,2026-10-27 00:00:00+00:00,80116.0,80364.2,80097.4,80193.3,2525.695,2026-10-27 00:59:59.999000+00:00,2.026508e+08,77278,1426.195,1.144323e+08
59785,2026-10-27 01:00:00+00:00,80193.2,80354.0,79996.0,80041.8,2591.985,2026-10-27 01:59:59.999000+00:00,2.077551e+08,104318,1235.100,9.900485e+07
59786,2026-10-27 02:00:00+00:00,80041.8,80091.3,79707.1,79789.7,4004.324,2026-10-27 02:59:59.999000+00:00,3.199792e+08,103863,1717.396,1.372355e+08
59787,2026-10-27 03:00:00+00:00,79789.6,79907.0,79600.0,79790.5,4019.331,2026-10-27 03:59:59.999000+00:00,3.205049e+08,97179,2113.133,1.685127e+08
